The goal of this notebook is to test lateral entrainment with sea surface temperature: if a target eddy takes in the water around it as it ages, its interior SST should move toward the SST of that water. SST is a physical tracer that no retrieval model produces, so it pairs with the plankton and pigment views of `eddy_evolution.ipynb`. The source is the MODIS-Aqua 8-day 4 km SST composite in `bronze/sst`, pixels with `qual_sst` of 0 or 1, from March 2024 to May 2026, so the record matches the PACE record and not the whole eddy record.

Target eddies:
- Cyclones formed north of the Gulf Stream axis and ended south, or formed within `NEAR_AXIS_KM` (150 km) of the axis and ended south. Anticyclones use the reversed rule.

Eddy requirements:
- For each composite, the track observation nearest the composite midpoint gives the center and the speed radius R. Pixels are binned by distance from the center in 0.2 R rings out to 3 R.
- The interior is 0 to 1 R and the local background is 2 to 3 R. Each needs 50% valid pixels and at least 10. A ring needs 3 valid pixels and 50%.
- Age is the fraction of the physical track, as in the gold tables. Within each age bin, the composites of each eddy are averaged, then the eddies of each polarity, with a bootstrap over eddies for the 95% interval.

Three references, each subtracted from the interior SST of the same composite:
- The 2 to 3 R ring around the eddy.
- The side of the axis the eddy ends on: the mean SST 150 to 450 km from the daily axis on that side, north for target anticyclones and south for target cyclones.
- The side the eddy came from: the same band on the other side.

Views:
- SST across the axis, the median and interquartile range over composites.
- The three anomalies against age, and the share of the first anomaly left by the last fifth of the track.
- The core, middle, and edge SST minus the surrounding water against age, with a radius-by-age heatmap, to see whether the edge reaches the surrounding water before the core.

An SST anomaly also decays by heat exchange with the air and by mixed-layer deepening, so a shrinking anomaly agrees with entrainment but does not prove it. The radial order is the part a surface flux alone does not give.

In [ ]:
from pathlib import Path
from typing import cast
import sys

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display
from cartopy.mpl.geoaxes import GeoAxes
from matplotlib.axes import Axes
from matplotlib.cm import ScalarMappable
from matplotlib.colors import BoundaryNorm, ListedColormap, Normalize
from matplotlib.figure import Figure
from matplotlib.ticker import MaxNLocator

PROJECT_ROOT = Path('/Users/jerry/school/research/eddy-tracking')
sys.path.insert(0, str(PROJECT_ROOT))
from eddy_tracking.config import load_config, resolve_data_dir
from eddy_tracking.packages.py_eddy_tracker.observations.tracking import TrackEddiesObservations
from eddy_tracking.preprocess.streamline import compute_signed_distance_grid_km, index_centerlines_by_date, trace_mean_streamline
from eddy_tracking.preprocess.tracks import PET_EPOCH

EXPERIMENT = 'gulf_stream_20240305_20260531'
N_AGE_BINS = 5
N_RADIAL_BINS = 15
MAX_RADIUS = 3
INTERIOR_RADIUS = 1
BACKGROUND_RADIUS = 2
MIN_COVERAGE = 0.5
MIN_PIXELS = 10
N_BOOTSTRAP = 2000
RANDOM_SEED = 2026
NEAR_AXIS_KM = 150
DISTANCE_BIN_KM = 50
MAX_DISTANCE_KM = 3 * NEAR_AXIS_KM
DATA_DIR = PROJECT_ROOT / 'data' / EXPERIMENT
polarity_names = ('cyclone', 'anticyclone')
target_classes = {'cyclone': 'NS', 'anticyclone': 'SN'}
polarity_colors = {'cyclone': '#2166ac', 'anticyclone': '#b2182b'}
target_labels = {'cyclone': 'Target cyclones', 'anticyclone': 'Target anticyclones'}
identity_columns = ['polarity', 'track_id']
references = ['local', 'destination', 'origin']
reference_labels = {'local': 'Minus surrounding water', 'destination': 'Minus water where it ends', 'origin': 'Minus water where it formed'}
zone_names = ('core', 'middle', 'edge')
zone_labels = {'core': 'Core (0 to 0.4 R)', 'middle': 'Middle (0.4 to 0.8 R)', 'edge': 'Edge (0.8 to 1.2 R)'}
RINGS_PER_ZONE = 2
N_HEAT_BINS = 10
panel_letters = 'abcdef'
bin_edges = np.linspace(0, 1, N_AGE_BINS + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
radial_edges = np.linspace(0, MAX_RADIUS, N_RADIAL_BINS + 1)
radial_centers = (radial_edges[:-1] + radial_edges[1:]) / 2
distance_edges = np.arange(-MAX_DISTANCE_KM, MAX_DISTANCE_KM + 1, DISTANCE_BIN_KM)
distance_centers = (distance_edges[:-1] + distance_edges[1:]) / 2
sides = {'south': (-MAX_DISTANCE_KM, -NEAR_AXIS_KM), 'north': (NEAR_AXIS_KM, MAX_DISTANCE_KM)}
quantile_levels = np.array([0.25, 0.5, 0.75])

centerline_by_date = index_centerlines_by_date(pd.read_parquet(DATA_DIR / 'silver/gulf_stream/streamline.parquet'))
eddy_tracks = pd.read_parquet(DATA_DIR / 'silver/gulf_stream/eddy_movement.parquet')
target_class = cast(pd.Series, eddy_tracks['polarity']).map(target_classes)
eddy_tracks['is_target'] = eddy_tracks['movement'].eq(target_class) | (
    eddy_tracks['birth_distance_km'].abs().le(NEAR_AXIS_KM) & eddy_tracks['death_side'].eq(target_class.str[1])
)
targets = eddy_tracks.loc[eddy_tracks['is_target'], identity_columns + ['birth_date', 'death_date']]
track_observations = []
for polarity in polarity_names:
    tracked = TrackEddiesObservations.load_file(str(DATA_DIR / f'silver/eddy_track/{polarity}/{polarity}_tracks.zarr'))
    keep = ~tracked.virtual.astype(bool)
    track_observations.append(pd.DataFrame({
        'polarity': polarity, 'track_id': tracked.track[keep].astype(int),
        'day': pd.Timestamp(PET_EPOCH) + pd.to_timedelta(tracked.time[keep].astype(int), unit='D'),
        'center_lon': (tracked.longitude[keep] + 180) % 360 - 180, 'center_lat': tracked.latitude[keep],
        'radius_km': tracked.radius_s[keep] / 1000,
    }))
target_observations = pd.concat(track_observations, ignore_index=True).merge(targets, on=identity_columns)

sst_files = sorted((DATA_DIR / 'bronze/sst').glob('*.nc'))
with xr.open_dataset(sst_files[0]) as ds:
    lon = ds['lon'].to_numpy()
    lat = ds['lat'].to_numpy()
composite_rows = []
sst_fields = []
for path in sst_files:
    start, end = (pd.Timestamp(part) for part in path.name.split('.')[1].split('_'))
    composite_rows.append({'start': start, 'end': end, 'date': start + cast(pd.Timedelta, end - start) / 2})
    with xr.open_dataset(path) as ds:
        sst_fields.append(ds['sst'].where(ds['qual_sst'] <= 1).to_numpy())
composites = pd.DataFrame(composite_rows)
sst_fields = np.stack(sst_fields)  # (n_composites, n_lat, n_lon)

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold',
    'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 150, 'savefig.dpi': 300,
    'legend.frameon': False,
})

display(pd.DataFrame({
    'composites': [len(composites)], 'first': [composites['start'].min().date()], 'last': [composites['end'].max().date()],
    'valid_pixels': [np.isfinite(sst_fields).mean().round(3)],
}))

## SST across the axis

In [ ]:
profile_rows = []
for composite in composites.itertuples():
    if composite.date.date() not in centerline_by_date:  # pyright: ignore[reportAttributeAccessIssue]
        continue
    centerline = centerline_by_date[composite.date.date()]  # pyright: ignore[reportAttributeAccessIssue]
    offset_km = compute_signed_distance_grid_km(centerline.lon, centerline.lat, lon, lat).ravel()  # (n_lat, n_lon) -> (n_lat * n_lon,)
    distance_bin = np.digitize(offset_km, distance_edges) - 1
    field = sst_fields[composite.Index].ravel()  # pyright: ignore[reportAttributeAccessIssue]
    valid = np.isfinite(offset_km) & (distance_bin >= 0) & (distance_bin < len(distance_centers)) & np.isfinite(field)
    sums = np.bincount(distance_bin[valid], weights=field[valid], minlength=len(distance_centers))
    counts = np.bincount(distance_bin[valid], minlength=len(distance_centers))
    profile_rows.append(pd.DataFrame({
        'date': composite.date, 'distance_km': distance_centers,  # pyright: ignore[reportAttributeAccessIssue]
        'sst': np.divide(sums, counts, out=np.full(len(distance_centers), np.nan), where=counts > 0),
    }))
sst_profile = pd.concat(profile_rows, ignore_index=True)
profile_quantiles = cast(pd.DataFrame, sst_profile.groupby('distance_km')['sst'].quantile(quantile_levels)).unstack()
side_sst = pd.DataFrame({
    side: sst_profile.loc[sst_profile['distance_km'].between(low, high)].groupby('date')['sst'].mean()
    for side, (low, high) in sides.items()
}).reset_index()

cfg = load_config(EXPERIMENT)
lon_range = cfg['base']['region']['lon_range']
lat_range = cfg['base']['region']['lat_range']
mean_axis = trace_mean_streamline(sorted(resolve_data_dir(cfg, 'swot_dir').glob('*.nc')), tuple(cfg['gulf_stream']['adt_level_range']))
valid_count = np.isfinite(sst_fields).sum(axis=0)
mean_sst = np.divide(np.nansum(sst_fields, axis=0), valid_count, out=np.full(valid_count.shape, np.nan), where=valid_count > 0)

profile_fig = plt.figure(figsize=(6.69, 2.63))
map_ax = cast(GeoAxes, profile_fig.add_axes((0.06, 0.14, 0.508, 0.775), projection=ccrs.PlateCarree()))
map_ax.set_extent([*lon_range, *lat_range], crs=ccrs.PlateCarree())
sst_fill = map_ax.contourf(lon, lat, mean_sst, levels=np.arange(8, 29.01, 1), cmap='plasma', transform=ccrs.PlateCarree(), zorder=1)
map_ax.plot(mean_axis.lon, mean_axis.lat, color='black', linewidth=1, transform=ccrs.PlateCarree(), zorder=4)
map_ax.add_feature(cfeature.LAND.with_scale('10m'), facecolor='#e8e8e8', zorder=2)
map_ax.coastlines(resolution='10m', color='#4d4d4d', linewidth=0.5, zorder=3)
gridlines = map_ax.gridlines(draw_labels=True, linewidth=0.5, color='#808080', alpha=0.5, xlocs=range(-80, -55, 5), ylocs=range(30, 45, 2), zorder=4)
gridlines.top_labels = False
gridlines.right_labels = False
sst_bar = profile_fig.colorbar(sst_fill, cax=map_ax.inset_axes((1.02, 0.1, 0.03, 0.8)), ticks=[10, 15, 20, 25])
sst_bar.set_label('SST (°C)')
map_ax.set_title(f'$\\bf{{(a)}}$ Mean SST and mean axis ({len(composites)} composites)', loc='left')

profile_ax = cast(Axes, profile_fig.add_axes((0.725, 0.14, 0.255, 0.775)))
profile_ax.axvline(0, color='#999999', linewidth=0.6, zorder=1)
profile_ax.fill_between(profile_quantiles.index, profile_quantiles[0.25], profile_quantiles[0.75], color='#444444', alpha=0.15, linewidth=0)  # pyright: ignore[reportArgumentType]
profile_ax.plot(profile_quantiles.index, profile_quantiles[0.5], color='#444444', linewidth=1.2)
profile_ax.set_xlim(-MAX_DISTANCE_KM, MAX_DISTANCE_KM)
profile_ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
profile_ax.set_axisbelow(True)
locator = MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10])
ticks = cast(np.ndarray, locator.tick_values(*profile_ax.get_ylim()))
profile_ax.yaxis.set_ticks(ticks)
profile_ax.set_ylim(ticks[0], ticks[-1])
profile_ax.set_title(f'$\\bf{{(b)}}$ SST across the axis (n = {sst_profile["date"].nunique()})', loc='left')
profile_ax.set_xlabel('Distance north of the axis (km)')
profile_ax.set_ylabel('SST (°C)')
plt.show()
display(side_sst.set_index('date').quantile(quantile_levels).round(2))

## Interior SST against its references over the lifetime

In [ ]:
rings = []
for composite in composites.itertuples():
    candidates = target_observations.loc[target_observations['day'].between(composite.start, composite.end)].copy()  # pyright: ignore[reportAttributeAccessIssue]
    candidates['offset'] = (candidates['day'] - composite.date).abs()  # pyright: ignore[reportAttributeAccessIssue]
    field = sst_fields[composite.Index]  # pyright: ignore[reportAttributeAccessIssue]
    for eddy in candidates.sort_values(['offset', 'day']).drop_duplicates(identity_columns).itertuples():
        half_width = MAX_RADIUS * eddy.radius_km / 111.32
        lon_index = np.flatnonzero(np.abs(lon - eddy.center_lon) <= half_width / np.cos(np.radians(eddy.center_lat)))
        lat_index = np.flatnonzero(np.abs(lat - eddy.center_lat) <= half_width)
        lon_grid, lat_grid = np.meshgrid(lon[lon_index], lat[lat_index])
        half_chord = (
            np.sin(np.radians(lat_grid - eddy.center_lat) / 2) ** 2
            + np.cos(np.radians(lat_grid)) * np.cos(np.radians(eddy.center_lat)) * np.sin(np.radians(lon_grid - eddy.center_lon) / 2) ** 2
        )
        distance_km = 2 * 6371 * np.arcsin(np.sqrt(half_chord))
        radial_bin = np.digitize(distance_km.ravel() / eddy.radius_km, radial_edges) - 1
        inside = radial_bin < N_RADIAL_BINS
        values = field[np.ix_(lat_index, lon_index)].ravel()
        valid = inside & np.isfinite(values)
        n_valid = np.bincount(radial_bin[valid], minlength=N_RADIAL_BINS)
        rings.append(pd.DataFrame({
            'polarity': eddy.polarity, 'track_id': eddy.track_id, 'date': composite.date, 'radial_bin': np.arange(N_RADIAL_BINS),  # pyright: ignore[reportAttributeAccessIssue]
            'n_pixels': np.bincount(radial_bin[inside], minlength=N_RADIAL_BINS), 'n_valid': n_valid,
            'sst': np.bincount(radial_bin[valid], weights=values[valid], minlength=N_RADIAL_BINS) / np.where(n_valid > 0, n_valid, np.nan),
        }))
eddy_rings = pd.concat(rings, ignore_index=True).merge(targets, on=identity_columns)
eddy_rings['age_frac'] = ((eddy_rings['date'] - eddy_rings['birth_date']).dt.days / (eddy_rings['death_date'] - eddy_rings['birth_date']).dt.days).clip(0, 1)
eddy_rings['age_bin'] = np.minimum((eddy_rings['age_frac'] * N_AGE_BINS).astype(int), N_AGE_BINS - 1)
eddy_rings['zone'] = np.select(
    [radial_edges[eddy_rings['radial_bin'] + 1] <= INTERIOR_RADIUS, radial_edges[eddy_rings['radial_bin']] >= BACKGROUND_RADIUS],
    ['interior', 'background'], default='',
)
zones = cast(pd.DataFrame, eddy_rings.loc[eddy_rings['zone'].ne('')].assign(sst_sum=eddy_rings['sst'].fillna(0) * eddy_rings['n_valid']).groupby(identity_columns + ['date', 'age_frac', 'age_bin', 'zone']).agg(
    sst_sum=('sst_sum', 'sum'), n_valid=('n_valid', 'sum'), n_pixels=('n_pixels', 'sum'),
)).reset_index()
zones['sst'] = np.where(zones['n_valid'].ge(MIN_PIXELS) & zones['n_valid'].ge(MIN_COVERAGE * zones['n_pixels']), zones['sst_sum'] / zones['n_valid'], np.nan)
anomalies = zones.pivot(index=identity_columns + ['date', 'age_frac', 'age_bin'], columns='zone', values='sst').reset_index().merge(side_sst, on='date')
destination = cast(pd.Series, anomalies['polarity']).map({'cyclone': 'south', 'anticyclone': 'north'})
anomalies['local'] = anomalies['interior'] - anomalies['background']
anomalies['destination'] = anomalies['interior'] - np.where(destination.eq('south'), anomalies['south'], anomalies['north'])
anomalies['origin'] = anomalies['interior'] - np.where(destination.eq('south'), anomalies['north'], anomalies['south'])
analysis = anomalies.melt(id_vars=identity_columns + ['date', 'age_frac', 'age_bin'], value_vars=references, var_name='reference', value_name='anomaly').dropna(subset=['anomaly'])
analysis = analysis.sort_values(['reference'] + identity_columns + ['date'], ignore_index=True)
analysis['initial'] = analysis.groupby(['reference'] + identity_columns)['anomaly'].transform('first')

eddy_bins = analysis.groupby(['reference'] + identity_columns + ['age_bin'])['anomaly'].mean().reset_index()
n_eddies = analysis.groupby(['reference', 'polarity'])['track_id'].nunique()
rng = np.random.default_rng(RANDOM_SEED)
summary_rows = []
for keys, members in eddy_bins.groupby(['reference', 'polarity']):
    reference, polarity = cast(tuple[str, str], keys)
    eddy_ids = sorted(members['track_id'].unique())
    matrix = members.pivot(index='track_id', columns='age_bin', values='anomaly').reindex(index=eddy_ids, columns=range(N_AGE_BINS)).to_numpy(dtype=float)
    counts = np.isfinite(matrix).sum(axis=0)
    means = np.divide(np.nansum(matrix, axis=0), counts, out=np.full(N_AGE_BINS, np.nan), where=counts > 0)
    sampled = matrix[rng.integers(0, len(eddy_ids), size=(N_BOOTSTRAP, len(eddy_ids)))]  # (n_eddies, n_bins) -> (n_bootstrap, n_eddies, n_bins)
    sampled_counts = np.isfinite(sampled).sum(axis=1)
    sampled_means = np.divide(np.nansum(sampled, axis=1), sampled_counts, out=np.full((N_BOOTSTRAP, N_AGE_BINS), np.nan), where=sampled_counts > 0)
    low, high = np.nanquantile(sampled_means, [0.025, 0.975], axis=0)
    summary_rows.append(pd.DataFrame({
        'reference': reference, 'polarity': polarity, 'age_bin': range(N_AGE_BINS), 'age_midpoint': bin_centers,
        'mean': means, 'ci_low': np.where(counts >= 3, low, np.nan), 'ci_high': np.where(counts >= 3, high, np.nan), 'n_eddies': counts,
    }))
lifetime_summary = pd.concat(summary_rows, ignore_index=True)

life_fig, life_axes = cast(tuple[Figure, np.ndarray], plt.subplots(1, 3, figsize=(6.69, 2.7), layout='constrained'))
for letter, ax, reference in zip(panel_letters, life_axes, references):
    ax = cast(Axes, ax)
    ax.axhline(0, color='#999999', linewidth=0.6, zorder=1)
    for polarity, color in polarity_colors.items():
        result = lifetime_summary.loc[lifetime_summary['reference'].eq(reference) & lifetime_summary['polarity'].eq(polarity)].sort_values('age_bin')
        intervals = result['ci_low'].notna()
        ax.errorbar(
            result.loc[intervals, 'age_midpoint'], result.loc[intervals, 'mean'],
            yerr=[result.loc[intervals, 'mean'] - result.loc[intervals, 'ci_low'], result.loc[intervals, 'ci_high'] - result.loc[intervals, 'mean']],
            fmt='none', ecolor=color, capsize=1.5, elinewidth=0.7, capthick=0.7, zorder=3,
        )
        ax.plot(result['age_midpoint'], result['mean'], '-o', color=color, linewidth=1.2, markersize=3.2, markeredgecolor='white', markeredgewidth=0.5, label=f'{target_labels[polarity]} (n = {n_eddies[reference, polarity]})', zorder=4)
    ax.set_title(f'$\\bf{{({letter})}}$ {reference_labels[reference]}', loc='left')
    ax.set_xlim(0, 1)
    ax.xaxis.set_ticks(np.linspace(0, 1, 6))
    ax.set_xlabel('Fraction of observed track')
    ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
    ax.set_axisbelow(True)
    locator = MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10])
    ticks = cast(np.ndarray, locator.tick_values(*ax.get_ylim()))
    ax.yaxis.set_ticks(ticks)
    ax.set_ylim(ticks[0], ticks[-1])
first_ax = cast(Axes, life_axes[0])
first_ax.set_ylabel('Interior SST minus reference (°C)')
life_fig.legend(*first_ax.get_legend_handles_labels(), loc='outside lower center', ncol=2)
plt.show()
display(lifetime_summary.pivot(index=['reference', 'polarity'], columns='age_bin', values='mean').reindex(references, level=0).round(2))
display(lifetime_summary.pivot(index=['reference', 'polarity'], columns='age_bin', values='n_eddies').reindex(references, level=0))

In [ ]:
final = analysis.loc[analysis['age_bin'].eq(N_AGE_BINS - 1) & analysis['reference'].ne('origin')].groupby(['reference'] + identity_columns).agg(final=('anomaly', 'mean'), initial=('initial', 'first')).reset_index()
rng = np.random.default_rng(RANDOM_SEED)
final_rows = []
for keys, members in final.groupby(['reference', 'polarity']):
    reference, polarity = cast(tuple[str, str], keys)
    initial = members['initial'].to_numpy()
    last = members['final'].to_numpy()
    change = (last - initial)[rng.integers(0, len(members), size=(N_BOOTSTRAP, len(members)))].mean(axis=1)  # (n_bootstrap, n_eddies) -> (n_bootstrap,)
    low, high = np.quantile(change, [0.025, 0.975])
    final_rows.append({
        'reference': reference, 'polarity': polarity, 'n_eddies': len(members),
        'initial': initial.mean(), 'final': last.mean(), 'change': (last - initial).mean(), 'ci_low': low, 'ci_high': high,
        'percent_remaining': 100 * last.mean() / initial.mean(), 'eddies_closer': int((np.abs(last) < np.abs(initial)).sum()),
    })
final_anomaly = pd.DataFrame(final_rows).set_index(['reference', 'polarity']).reindex(references[:2], level=0).reset_index()
display(final_anomaly.round(2))

## SST anomaly by zone and age

In [ ]:
zone_rings = eddy_rings.loc[eddy_rings['radial_bin'].lt(len(zone_names) * RINGS_PER_ZONE)].assign(
    zone=lambda rings: np.array(zone_names)[rings['radial_bin'] // RINGS_PER_ZONE], sst_sum=lambda rings: rings['sst'].fillna(0) * rings['n_valid'],
)
zone_sst = cast(pd.DataFrame, zone_rings.groupby(identity_columns + ['date', 'age_frac', 'age_bin', 'zone']).agg(
    sst_sum=('sst_sum', 'sum'), n_valid=('n_valid', 'sum'), n_pixels=('n_pixels', 'sum'),
)).reset_index().merge(anomalies[identity_columns + ['date', 'background']], on=identity_columns + ['date'])
covered = zone_sst['n_valid'].ge(3) & zone_sst['n_valid'].ge(MIN_COVERAGE * zone_sst['n_pixels'])
zone_sst['anomaly'] = (zone_sst['sst_sum'] / zone_sst['n_valid'] - zone_sst['background']).where(covered)
zone_bins = zone_sst.dropna(subset=['anomaly']).groupby(identity_columns + ['zone', 'age_bin'])['anomaly'].mean().reset_index()
rng = np.random.default_rng(RANDOM_SEED)
zone_rows = []
for keys, members in zone_bins.groupby(['polarity', 'zone']):
    polarity, zone = cast(tuple[str, str], keys)
    eddy_ids = sorted(members['track_id'].unique())
    matrix = members.pivot(index='track_id', columns='age_bin', values='anomaly').reindex(index=eddy_ids, columns=range(N_AGE_BINS)).to_numpy(dtype=float)
    counts = np.isfinite(matrix).sum(axis=0)
    means = np.divide(np.nansum(matrix, axis=0), counts, out=np.full(N_AGE_BINS, np.nan), where=counts > 0)
    sampled = matrix[rng.integers(0, len(eddy_ids), size=(N_BOOTSTRAP, len(eddy_ids)))]  # (n_eddies, n_bins) -> (n_bootstrap, n_eddies, n_bins)
    sampled_counts = np.isfinite(sampled).sum(axis=1)
    sampled_means = np.divide(np.nansum(sampled, axis=1), sampled_counts, out=np.full((N_BOOTSTRAP, N_AGE_BINS), np.nan), where=sampled_counts > 0)
    low, high = np.nanquantile(sampled_means, [0.025, 0.975], axis=0)
    zone_rows.append(pd.DataFrame({
        'polarity': polarity, 'zone': zone, 'age_bin': range(N_AGE_BINS), 'age_midpoint': bin_centers,
        'mean': means, 'ci_low': np.where(counts >= 3, low, np.nan), 'ci_high': np.where(counts >= 3, high, np.nan), 'n_eddies': counts,
    }))
zone_summary = pd.concat(zone_rows, ignore_index=True)
n_zone_eddies = zone_bins.groupby('polarity')['track_id'].nunique()

ring_anomaly = eddy_rings.merge(anomalies[identity_columns + ['date', 'background']], on=identity_columns + ['date'])
covered = ring_anomaly['n_valid'].ge(3) & ring_anomaly['n_valid'].ge(MIN_COVERAGE * ring_anomaly['n_pixels'])
ring_anomaly['anomaly'] = (ring_anomaly['sst'] - ring_anomaly['background']).where(covered)
ring_cells = cast(pd.DataFrame, ring_anomaly.groupby(identity_columns + ['age_bin', 'radial_bin'])['anomaly'].mean().reset_index().groupby(['polarity', 'age_bin', 'radial_bin']).agg(
    anomaly=('anomaly', 'mean'), n_eddies=('anomaly', 'count'),
)).reset_index()
ring_cells.loc[ring_cells['n_eddies'].lt(3), 'anomaly'] = np.nan
heat_cells = ring_cells.loc[ring_cells['radial_bin'].lt(N_HEAT_BINS)]

zone_cmaps = {'anticyclone': plt.get_cmap('Reds')(np.linspace(0.95, 0.35, len(zone_names))), 'cyclone': plt.get_cmap('Blues')(np.linspace(0.95, 0.35, len(zone_names)))}
heat_cmap = plt.get_cmap('RdBu_r').copy()
heat_cmap.set_bad('#e6e6e6')
heat_limit = heat_cells['anomaly'].abs().max()
heat_norm = Normalize(-heat_limit, heat_limit)
zone_fig = cast(Figure, plt.figure(figsize=(6.69, 5.0)))
outer = zone_fig.add_gridspec(2, 2, left=0.09, right=0.9, bottom=0.08, top=0.95, hspace=0.45, wspace=0.3)
line_axes = [cast(Axes, zone_fig.add_subplot(outer[0, index])) for index in range(2)]
heat_axes = [cast(Axes, zone_fig.add_subplot(outer[1, index])) for index in range(2)]
for letter, ax, polarity in zip(panel_letters, line_axes, ('anticyclone', 'cyclone')):
    ax.axhline(0, color='#999999', linewidth=0.6, zorder=1)
    for zone, color in zip(zone_names, zone_cmaps[polarity]):
        result = zone_summary.loc[zone_summary['polarity'].eq(polarity) & zone_summary['zone'].eq(zone)].sort_values('age_bin')
        ax.fill_between(result['age_midpoint'], result['ci_low'], result['ci_high'], color=color, alpha=0.15, linewidth=0, zorder=2)  # pyright: ignore[reportArgumentType]
        ax.plot(result['age_midpoint'], result['mean'], color=color, linewidth=1.4, label=zone_labels[zone], zorder=3)
    ax.set_title(f'$\\bf{{({letter})}}$ {target_labels[polarity]} (n = {n_zone_eddies[polarity]})', loc='left')
    ax.set_xlim(0, 1)
    ax.xaxis.set_ticks(np.linspace(0, 1, 6))
    ax.set_xlabel('Fraction of observed track')
    ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
    ax.set_axisbelow(True)
    locator = MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10])
    ticks = cast(np.ndarray, locator.tick_values(*ax.get_ylim()))
    ax.yaxis.set_ticks(ticks)
    ax.set_ylim(ticks[0], ticks[-1])
    ax.legend(loc='upper left' if polarity == 'anticyclone' else 'lower right', handlelength=1.8)
    ax.text(0.98, 0, 'same as surrounding water', color='#808080', ha='right', va='bottom', transform=ax.get_yaxis_transform(), zorder=4)
line_axes[0].set_ylabel('Eddy SST minus surrounding SST (°C)')
for letter, ax, polarity in zip(panel_letters[2:], heat_axes, ('anticyclone', 'cyclone')):
    grid = heat_cells.loc[heat_cells['polarity'].eq(polarity)].pivot(index='radial_bin', columns='age_bin', values='anomaly').reindex(index=range(N_HEAT_BINS), columns=range(N_AGE_BINS)).to_numpy(dtype=float)
    ax.pcolormesh(bin_edges, radial_edges[:N_HEAT_BINS + 1], np.ma.masked_invalid(grid), cmap=heat_cmap, norm=heat_norm, edgecolors='white', linewidth=0.5)
    ax.axhline(1, color='#222222', linewidth=0.8, linestyle=(0, (4, 2.5)), zorder=3)
    ax.set_title(f'$\\bf{{({letter})}}$ {target_labels[polarity]} by radius and age', loc='left')
    ax.xaxis.set_ticks(np.linspace(0, 1, 6))
    ax.yaxis.set_ticks([0, 0.5, 1, 1.5, 2], ['0', '0.5', '1', '1.5', '2'])
    ax.yaxis.set_ticks(radial_edges[:N_HEAT_BINS + 1], minor=True)
    ax.tick_params(length=2)
    ax.tick_params(which='minor', length=1.2)
    ax.set_xlabel('Fraction of observed track')
heat_axes[0].set_ylabel('Distance from center (R)')
heat_bar = zone_fig.colorbar(ScalarMappable(norm=heat_norm, cmap=heat_cmap), cax=heat_axes[1].inset_axes((1.05, 0, 0.04, 1)))
heat_bar.ax.tick_params(length=2)
heat_bar.ax.yaxis.set_major_locator(MaxNLocator(nbins=4, steps=[1, 2, 2.5, 5, 10]))
heat_bar.outline.set_linewidth(0.5)
heat_bar.set_label('Eddy SST minus surrounding SST (°C)')
plt.show()
display(zone_summary.pivot(index=['polarity', 'zone'], columns='age_bin', values='mean').reindex(zone_names, level=1).round(2))
display(zone_summary.pivot(index=['polarity', 'zone'], columns='age_bin', values='n_eddies').reindex(zone_names, level=1))
display(heat_cells.pivot(index=['polarity', 'age_bin'], columns='radial_bin', values='anomaly').round(2))